# Part A Question 2: Word Stemming

This notebook demonstrates stemming with a regular expression stemmer, Porter Stemmer, and Lancaster Stemmer using `Data_1.txt`.

In [8]:
import re
import string
from pathlib import Path
from pprint import pprint

import matplotlib.pyplot as plt
import nltk
import pandas as pd
from nltk.data import find
from nltk.stem import LancasterStemmer, PorterStemmer, RegexpStemmer
from nltk.tokenize import word_tokenize

In [9]:
DATA_PATH = Path('data/Data_1.txt')
if not DATA_PATH.exists():
    DATA_PATH = Path('../data/Data_1.txt')


def load_text(path: Path) -> str:
    """Read a UTF-8 text file and normalize line breaks to spaces."""

    return path.read_text(encoding='utf-8').replace('\r\n', '\n').replace('\n', ' ')


text = load_text(DATA_PATH)
text

'Classification is the task of choosing the correct class label for a given input. In basic classification tasks, each input is considered in isolation from all other inputs, and the set of labels is defined in advance. The basic classification task has a number of interesting variants. For example, in multiclass classification, each instance may be assigned multiple labels; in open-class classification, the set of labels is not defined in advance; and in sequence classification, a list of inputs are jointly classified.'

## 1. Pre-processing

In [10]:
def tokenize_with_nltk(text: str) -> list[str]:
    """Tokenize with NLTK's word tokenizer using Punkt sentence rules."""

    try:
        find('tokenizers/punkt_tab')
    except LookupError:
        nltk.download('punkt_tab', quiet=True)

    return word_tokenize(text)


def get_stopwords() -> set[str]:
    """Return NLTK English stop words, downloading the corpus if needed."""

    try:
        find('corpora/stopwords')
    except LookupError:
        nltk.download('stopwords', quiet=True)

    from nltk.corpus import stopwords

    return set(stopwords.words('english'))


def content_tokens(text: str) -> list[str]:
    """Return punctuation-free content tokens for stemming demonstrations."""

    stopwords = get_stopwords()
    punctuation = set(string.punctuation)
    tokens: list[str] = []

    for token in tokenize_with_nltk(text):
        cleaned_token = token.strip(string.punctuation)
        if not cleaned_token:
            continue
        if cleaned_token.casefold() in stopwords:
            continue
        tokens.append(cleaned_token)

    return tokens


tokens = content_tokens(text)
pprint(tokens)

['Classification',
 'task',
 'choosing',
 'correct',
 'class',
 'label',
 'given',
 'input',
 'basic',
 'classification',
 'tasks',
 'input',
 'considered',
 'isolation',
 'inputs',
 'set',
 'labels',
 'defined',
 'advance',
 'basic',
 'classification',
 'task',
 'number',
 'interesting',
 'variants',
 'example',
 'multiclass',
 'classification',
 'instance',
 'may',
 'assigned',
 'multiple',
 'labels',
 'open-class',
 'classification',
 'set',
 'labels',
 'defined',
 'advance',
 'sequence',
 'classification',
 'list',
 'inputs',
 'jointly',
 'classified']


## 2. Regular Expression Stemmer

In [11]:
regex_stemmer = RegexpStemmer(r'(ing|ly|ed|ious|ies|ive|es|s|ment)$', min=4)
regex_stems = [regex_stemmer.stem(token) for token in tokens]
pprint(regex_stems)

['Classification',
 'task',
 'choos',
 'correct',
 'clas',
 'label',
 'given',
 'input',
 'basic',
 'classification',
 'task',
 'input',
 'consider',
 'isolation',
 'input',
 'set',
 'label',
 'defin',
 'advance',
 'basic',
 'classification',
 'task',
 'number',
 'interest',
 'variant',
 'example',
 'multiclas',
 'classification',
 'instance',
 'may',
 'assign',
 'multiple',
 'label',
 'open-clas',
 'classification',
 'set',
 'label',
 'defin',
 'advance',
 'sequence',
 'classification',
 'list',
 'input',
 'joint',
 'classifi']


## 3. Porter Stemmer

In [12]:
porter_stemmer = PorterStemmer()
porter_stems = [porter_stemmer.stem(token) for token in tokens]
pprint(porter_stems)

['classif',
 'task',
 'choos',
 'correct',
 'class',
 'label',
 'given',
 'input',
 'basic',
 'classif',
 'task',
 'input',
 'consid',
 'isol',
 'input',
 'set',
 'label',
 'defin',
 'advanc',
 'basic',
 'classif',
 'task',
 'number',
 'interest',
 'variant',
 'exampl',
 'multiclass',
 'classif',
 'instanc',
 'may',
 'assign',
 'multipl',
 'label',
 'open-class',
 'classif',
 'set',
 'label',
 'defin',
 'advanc',
 'sequenc',
 'classif',
 'list',
 'input',
 'jointli',
 'classifi']


## 4. Lancaster Stemmer

In [13]:
lancaster_stemmer = LancasterStemmer()
lancaster_stems = [lancaster_stemmer.stem(token) for token in tokens]
pprint(lancaster_stems)

['class',
 'task',
 'choos',
 'correct',
 'class',
 'label',
 'giv',
 'input',
 'bas',
 'class',
 'task',
 'input',
 'consid',
 'isol',
 'input',
 'set',
 'label',
 'defin',
 'adv',
 'bas',
 'class',
 'task',
 'numb',
 'interest',
 'vary',
 'exampl',
 'multiclass',
 'class',
 'inst',
 'may',
 'assign',
 'multipl',
 'label',
 'open-class',
 'class',
 'set',
 'label',
 'defin',
 'adv',
 'sequ',
 'class',
 'list',
 'input',
 'joint',
 'class']


## 5. Comparison Table

In [14]:
comparison_table = pd.DataFrame(
    {
        'Original Token': tokens,
        'Regex Stemmer': regex_stems,
        'Porter Stemmer': porter_stems,
        'Lancaster Stemmer': lancaster_stems,
    }
)

comparison_table

,Original Token,Regex Stemmer,Porter Stemmer,Lancaster Stemmer
0,Classification,Classification,classif,class
1,task,task,task,task
2,choosing,choos,choos,choos
3,correct,correct,correct,correct
4,class,clas,class,class
5,label,label,label,label
6,given,given,given,giv
7,input,input,input,input
8,basic,basic,basic,bas
9,classification,classification,classif,class


## 6. Save Comparison Table as PNG

In [ ]:
OUTPUT_PATH = Path('outputs/question_2_stemming_comparison.png')
if not Path('data/Data_1.txt').exists():
    OUTPUT_PATH = Path('../outputs/question_2_stemming_comparison.png')
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

figure_height = max(4, 0.35 * len(comparison_table))
fig, ax = plt.subplots(figsize=(12, figure_height))
ax.axis('off')

table = ax.table(
    cellText=comparison_table.values,
    colLabels=comparison_table.columns,
    cellLoc='left',
    colLoc='left',
    loc='center',
)
table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1, 1.25)

for (row, column), cell in table.get_celld().items():
    if row == 0:
        cell.set_text_props(weight='bold')
        cell.set_facecolor('#EAEAEA')

fig.tight_layout()
fig.savefig(OUTPUT_PATH, dpi=200, bbox_inches='tight')
plt.close(fig)

OUTPUT_PATH
